# 🏀 NBA Projections & Insight Ledger

Three models, each forecasting **both** the total and the home margin, blended by Bayesian Predictive
Synthesis, with Brier skill, confusion matrices and the Money Ball bet rules (expected value or credible
interval). Data is read from the [vladk-III/NBA](https://github.com/vladk-III/NBA) repo, which the GitHub
Actions pipeline updates every day.

**How to use:** Runtime → Run all. The last cell opens the dashboard. Daily, after the overnight data run:
**1 Train → 2 Optimize BPS → 3 Predict**. Model state and `predictions_history.csv` are saved to Google Drive
under `Stonks/Basketball/NBA/2026-27`.

*Generated from `model/*.py` by `model/build_colab.py`; edit the .py files and rebuild rather than editing
this notebook.*

In [ ]:
!pip install -q pymc nutpie arviz netCDF4 dash plotly diskcache multiprocess requests

## Core: features, Bayesian Predictive Synthesis, odds and grading

In [ ]:
"""
Pure numpy/pandas pieces of the NBA modeler (no PyMC, no Dash), so they can be unit-tested
and reused: features, Bayesian Predictive Synthesis filter, odds handling and grading.

Everything here works on the files the data pipeline commits to data/:
  historical_nba_cumulative_stats.csv  pre-game season-to-date team stats, one row per game
  odds_snapshots.csv                   every US book's lines, morning and closing snapshots
"""


import numpy as np
import pandas as pd

# =============================================================================
# FEATURES
# =============================================================================
# League per-game averages used when no earlier season is available to estimate them.
LEAGUE_DEFAULTS = {"pts": 115.0, "poss": 99.5}
# Early-season shrinkage: a team's average after g games is blended with the league average
# as (g * team + k * league) / (g + k). With k = 10, a team's own numbers get half the weight
# after 10 games and about 85% by game 55.
SHRINK_GAMES = 10.0
REST_CAP = 4  # rest days beyond this add nothing


def side_stats(hist: pd.DataFrame, side: str) -> pd.DataFrame:
    """Per-game averages for one side, derived only from the original 50 columns so every
    season (including the 2025-26 rows from before the pipeline) gets the same features.

    opp_pts = PTS/G - plus,  oreb = REB/G - D_REB/G,  poss = FGA - OREB + TOV + 0.44 * FTA.
    All are averages of per-game values, and the formulas are linear, so they are exact.
    """
    p = f"{side}_"
    out = pd.DataFrame(index=hist.index)
    out["games"] = pd.to_numeric(hist[p + "games"], errors="coerce").fillna(0)
    out["pts"] = hist[p + "PTS/G"]
    out["opp_pts"] = hist[p + "PTS/G"] - hist[p + "plus"]
    oreb = hist[p + "REB/G"] - hist[p + "D_REB/G"]
    out["poss"] = hist[p + "FGA/G"] - oreb + hist[p + "TO/G"] + 0.44 * hist[p + "FTA/G"]
    fga = hist[p + "FGA/G"].where(hist[p + "FGA/G"] > 0)
    out["fg3a_rate"] = hist[p + "FG3A/G"] / fga
    out["ft_rate"] = hist[p + "FTA/G"] / fga
    return out.astype(float)


def league_prior(hist: pd.DataFrame, min_games: int = 20) -> dict:
    """League-average points and possessions per team-game from rows where teams have played
    at least min_games (so early-season noise doesn't set the prior). Falls back to defaults."""
    vals = {"pts": [], "poss": [], "fg3a_rate": [], "ft_rate": []}
    for side in ("home", "away"):
        s = side_stats(hist, side)
        s = s[(s["games"] >= min_games) & (s["poss"] > 0)]
        for k in vals:
            vals[k].append(s[k])
    prior = {}
    for k, parts in vals.items():
        series = pd.concat(parts) if parts else pd.Series(dtype=float)
        prior[k] = float(series.mean()) if len(series) else np.nan
    prior["pts"] = prior["pts"] if np.isfinite(prior["pts"]) else LEAGUE_DEFAULTS["pts"]
    prior["poss"] = prior["poss"] if np.isfinite(prior["poss"]) else LEAGUE_DEFAULTS["poss"]
    prior["fg3a_rate"] = prior["fg3a_rate"] if np.isfinite(prior["fg3a_rate"]) else 0.42
    prior["ft_rate"] = prior["ft_rate"] if np.isfinite(prior["ft_rate"]) else 0.26
    prior["rtg"] = prior["pts"] / prior["poss"] * 100.0
    return prior


def shrink(value, games, prior_value, k: float = SHRINK_GAMES):
    value = pd.Series(value, dtype=float)
    games = pd.Series(games, dtype=float).fillna(0).clip(lower=0)
    filled = value.where(games > 0, prior_value).fillna(prior_value)
    return (games * filled + k * prior_value) / (games + k)


def rest_days_from_schedule(hist: pd.DataFrame) -> pd.DataFrame:
    """Days since each side's previous game in the same season, computed from the history file
    itself (so 2025-26 rows get it too). First game of a season -> NaN."""
    rows = []
    for side in ("home", "away"):
        rows.append(pd.DataFrame({"idx": hist.index, "season": hist["season"].values,
                                  "team": hist[f"{side}_team_abbreviation"].values,
                                  "date": pd.to_datetime(hist["game_date"]).values, "side": side}))
    long = pd.concat(rows, ignore_index=True).sort_values(["season", "team", "date"])
    long["prev"] = long.groupby(["season", "team"])["date"].shift(1)
    long["rest"] = (long["date"] - long["prev"]).dt.days
    wide = long.pivot_table(index="idx", columns="side", values="rest", aggfunc="first")
    return wide.reindex(hist.index).rename(columns={"home": "home_rest", "away": "away_rest"})


def build_features(hist: pd.DataFrame, prior: dict | None = None, k: float = SHRINK_GAMES) -> pd.DataFrame:
    """One row per game: shrunk pace/efficiency for each side, the pace x efficiency
    expectation for the score, rest, and the actual result when the game is final."""
    hist = hist.reset_index(drop=True)
    if "season" not in hist.columns:
        hist = hist.assign(season="unknown")
    prior = prior or league_prior(hist)
    df = pd.DataFrame({
        "game_id": hist["game_id"].astype(str),
        "game_date": hist["game_date"].astype(str).str[:10],
        "season": hist["season"].astype(str),
        "home_team": hist["home_team_abbreviation"],
        "away_team": hist["away_team_abbreviation"],
        "home_points": pd.to_numeric(hist["home_points"], errors="coerce"),
        "away_points": pd.to_numeric(hist["away_points"], errors="coerce"),
    })
    for side in ("home", "away"):
        s = side_stats(hist, side)
        g = s["games"]
        pace = shrink(s["poss"], g, prior["poss"], k)
        pts = shrink(s["pts"], g, prior["pts"], k)
        opp = shrink(s["opp_pts"], g, prior["pts"], k)
        df[f"{side}_games"] = g.values
        df[f"{side}_pace"] = pace.values
        df[f"{side}_ortg"] = (pts / pace * 100).values
        df[f"{side}_drtg"] = (opp / pace * 100).values
        df[f"{side}_fg3a_rate"] = shrink(s["fg3a_rate"], g, prior["fg3a_rate"], k).values
        df[f"{side}_ft_rate"] = shrink(s["ft_rate"], g, prior["ft_rate"], k).values

    rest = rest_days_from_schedule(hist)
    for side in ("home", "away"):
        r = rest[f"{side}_rest"]
        df[f"{side}_b2b"] = (r == 1).astype(float).values
        df[f"{side}_rest"] = r.fillna(REST_CAP).clip(upper=REST_CAP).values
    df["rest_diff"] = df["home_rest"] - df["away_rest"]

    # Pace x efficiency expectation (no home court; models learn that themselves).
    lg_pace, lg_rtg = prior["poss"], prior["rtg"]
    exp_poss = df["home_pace"] * df["away_pace"] / lg_pace
    df["exp_poss"] = exp_poss
    df["exp_home"] = exp_poss * df["home_ortg"] * df["away_drtg"] / lg_rtg / 100.0
    df["exp_away"] = exp_poss * df["away_ortg"] * df["home_drtg"] / lg_rtg / 100.0
    df["exp_total"] = df["exp_home"] + df["exp_away"]
    df["exp_margin"] = df["exp_home"] - df["exp_away"]

    df["total_points"] = df["home_points"] + df["away_points"]
    df["margin"] = df["home_points"] - df["away_points"]
    return df


# =============================================================================
# ODDS
# =============================================================================
def american_to_prob(odds):
    o = pd.to_numeric(pd.Series(odds), errors="coerce")
    return pd.Series(np.where(o < 0, -o / (-o + 100.0), 100.0 / (o + 100.0)), index=o.index).where(o.notna())


def american_to_decimal(odds):
    o = pd.to_numeric(pd.Series(odds), errors="coerce")
    return pd.Series(np.where(o < 0, 1.0 + 100.0 / -o, 1.0 + o / 100.0), index=o.index).where(o.notna())


def devig_pair(price_a, price_b):
    """Fair probability of side A from the two prices of one market (multiplicative de-vig)."""
    pa, pb = american_to_prob(price_a), american_to_prob(price_b)
    return pa / (pa + pb)


def market_lines(odds: pd.DataFrame, label: str = "morning") -> pd.DataFrame:
    """One row per game_id from a snapshot label: consensus (median) total and home spread,
    the median price at that consensus line, and the best price any book offers at it."""
    cols = ["game_id", "line_total", "line_spread_home", "over_price", "under_price",
            "best_over_price", "best_under_price", "home_price", "away_price",
            "best_home_price", "best_away_price", "n_books"]
    if odds is None or odds.empty:
        return pd.DataFrame(columns=cols)
    o = odds[(odds["snapshot_label"] == label) & odds["game_id"].notna()].copy()
    if o.empty:
        return pd.DataFrame(columns=cols)
    o["game_id"] = o["game_id"].astype(str)
    last = o.groupby("game_id")["snapshot_utc"].transform("max")
    o = o[o["snapshot_utc"] == last]
    for c in ["total", "spread_home", "over_price", "under_price", "spread_home_price", "spread_away_price"]:
        o[c] = pd.to_numeric(o[c], errors="coerce")

    rows = []
    for gid, g in o.groupby("game_id"):
        row = {"game_id": gid, "n_books": int(g["bookmaker"].nunique())}
        t = g.dropna(subset=["total"])
        row["line_total"] = float(t["total"].median()) if len(t) else np.nan
        at = t[t["total"] == t["total"].median()] if len(t) else t
        if at.empty and len(t):  # median falls between two lines: use the most common one
            at = t[t["total"] == t["total"].mode().iloc[0]]
            row["line_total"] = float(at["total"].iloc[0])
        row["over_price"] = float(at["over_price"].median()) if len(at) else np.nan
        row["under_price"] = float(at["under_price"].median()) if len(at) else np.nan
        row["best_over_price"] = _best(at["over_price"])
        row["best_under_price"] = _best(at["under_price"])
        s = g.dropna(subset=["spread_home"])
        row["line_spread_home"] = float(s["spread_home"].median()) if len(s) else np.nan
        ats = s[s["spread_home"] == s["spread_home"].median()] if len(s) else s
        if ats.empty and len(s):
            ats = s[s["spread_home"] == s["spread_home"].mode().iloc[0]]
            row["line_spread_home"] = float(ats["spread_home"].iloc[0])
        row["home_price"] = float(ats["spread_home_price"].median()) if len(ats) else np.nan
        row["away_price"] = float(ats["spread_away_price"].median()) if len(ats) else np.nan
        row["best_home_price"] = _best(ats["spread_home_price"])
        row["best_away_price"] = _best(ats["spread_away_price"])
        rows.append(row)
    return pd.DataFrame(rows, columns=cols)


def _best(prices: pd.Series) -> float:
    """The price that pays most (highest decimal odds), as American odds."""
    p = pd.to_numeric(prices, errors="coerce").dropna()
    if p.empty:
        return np.nan
    return float(p.iloc[int(np.argmax(american_to_decimal(p).to_numpy()))])


# =============================================================================
# BAYESIAN PREDICTIVE SYNTHESIS — Rao-Blackwellized particle filter, K agents
# =============================================================================
# Same model as the NFL dashboard (McAlinn, Aastveit, Nakajima & West 2018), generalized from
# 2 agents to K. Synthesis: y_t = theta_t' x_t + nu_t,  x_t = (1, x_1t, ..., x_Kt), one draw
# from each agent's forecast density N(mean_k, sd_k). theta_t is a random walk discounted by
# delta (R_t = C_{t-1}/delta); the residual variance is discounted by beta. Each particle keeps
# an analytic Kalman state for theta; particles are reweighted by the one-step predictive
# likelihood and systematically resampled when the ESS drops.
BPS_DEFAULTS = {"particles": 1000, "samples": 2000, "delta": 0.99, "beta": 0.99}
BPS_PRIOR_DOF = 7.0
BPS_ESS_FRAC = 0.5
BPS_PRIOR_INTERCEPT_SD = 5.0
BPS_PRIOR_WEIGHT_SD = 0.25
BPS_FALLBACK_OBS_VAR = {"total": 300.0, "margin": 150.0}


class BPSFilter:
    def __init__(self, n_agents: int, n_particles: int, delta: float = 0.99, beta: float = 0.99,
                 obs_var0: float = 300.0, seed: int = 20261021):
        self.k = int(n_agents)
        self.dim = self.k + 1
        self.n = int(n_particles)
        self.delta, self.beta = float(delta), float(beta)
        self.rng = np.random.default_rng(seed)
        m0 = np.concatenate([[0.0], np.full(self.k, 1.0 / self.k)])
        self.m = np.tile(m0, (self.n, 1))
        C0 = np.diag([BPS_PRIOR_INTERCEPT_SD ** 2] + [BPS_PRIOR_WEIGHT_SD ** 2] * self.k)
        self.C = np.tile(C0, (self.n, 1, 1))
        self.h = np.full(self.n, BPS_PRIOR_DOF)
        self.D = np.full(self.n, BPS_PRIOR_DOF * float(obs_var0))
        self.w = np.full(self.n, 1.0 / self.n)

    def _draw_x(self, mu, sd, n):
        z = self.rng.standard_normal((n, self.k))
        X = np.ones((n, self.dim))
        X[:, 1:] = np.asarray(mu, float) + np.asarray(sd, float) * z
        return X

    @property
    def ess(self) -> float:
        return float(1.0 / np.sum(self.w ** 2))

    def theta_mean(self) -> np.ndarray:
        return self.w @ self.m

    def vol_mean(self) -> float:
        return float(np.sqrt(self.w @ (self.D / self.h)))

    def update(self, mu, sd, y: float) -> None:
        X = self._draw_x(mu, sd, self.n)
        R = self.C / self.delta
        f = np.einsum("ni,ni->n", X, self.m)
        RX = np.einsum("nij,nj->ni", R, X)
        S = self.D / self.h
        Q = np.einsum("ni,ni->n", X, RX) + S
        e = y - f
        loglik = -0.5 * (np.log(2 * np.pi * Q) + e ** 2 / Q)
        logw = np.log(self.w + 1e-300) + loglik
        logw -= logw.max()
        w = np.exp(logw)
        self.w = w / w.sum()

        K = RX / Q[:, None]
        self.m = self.m + K * e[:, None]
        C = R - K[:, :, None] * RX[:, None, :]
        self.C = 0.5 * (C + C.transpose(0, 2, 1))
        self.h = self.beta * self.h + 1.0
        self.D = self.beta * self.D + S * e ** 2 / Q

        if self.ess < BPS_ESS_FRAC * self.n:
            pos = (self.rng.random() + np.arange(self.n)) / self.n
            cs = np.cumsum(self.w)
            cs[-1] = 1.0
            idx = np.searchsorted(cs, pos)
            self.m, self.C, self.h, self.D = self.m[idx], self.C[idx], self.h[idx], self.D[idx]
            self.w = np.full(self.n, 1.0 / self.n)

    def predict(self, mu, sd, n_draws: int) -> dict:
        idx = self.rng.choice(self.n, size=int(n_draws), p=self.w / self.w.sum())
        X = self._draw_x(mu, sd, int(n_draws))
        R = self.C[idx] / self.delta
        f = np.einsum("di,di->d", X, self.m[idx])
        Q = np.einsum("di,dij,dj->d", X, R, X) + self.D[idx] / self.h[idx]
        draws = f + np.sqrt(Q) * self.rng.standard_normal(int(n_draws))
        return {"draws": draws, "mean": float(f.mean()), "var": float(Q.mean() + f.var())}

    def get_state(self) -> dict:
        return {"k": self.k, "n": self.n, "delta": self.delta, "beta": self.beta,
                "m": self.m, "C": self.C, "h": self.h, "D": self.D, "w": self.w}

    @classmethod
    def from_state(cls, state: dict, seed: int = 20261021) -> "BPSFilter":
        obj = cls(state["k"], state["n"], state["delta"], state["beta"], seed=seed)
        obj.m, obj.C, obj.h, obj.D, obj.w = state["m"], state["C"], state["h"], state["D"], state["w"]
        return obj


def fit_bps(df: pd.DataFrame, mu_cols: list[str], sd_cols: list[str], actual_col: str,
            n_particles: int, delta: float, beta: float, seed: int) -> dict:
    """Run the filter forward through graded games in date order."""
    d = df.dropna(subset=mu_cols + [actual_col]).copy()
    if len(d) < 3:
        raise RuntimeError(f"Only {len(d)} graded game(s) with forecasts from every model; BPS needs at least 3.")
    d = d.sort_values(["game_date", "game_id"])
    y = d[actual_col].to_numpy(float)
    MU = d[mu_cols].to_numpy(float)
    fallback_sd = np.sqrt(np.mean((MU - y[:, None]) ** 2, axis=0))
    SD = np.column_stack([
        np.where(np.isfinite(v := d[c].to_numpy(float)) & (v > 0), v, fallback_sd[i])
        for i, c in enumerate(sd_cols)])
    obs_var0 = max(0.5 * float(np.var(y - MU.mean(axis=1))), 1e-3)
    flt = BPSFilter(len(mu_cols), n_particles, delta, beta, obs_var0, seed)
    for t in range(len(y)):
        flt.update(MU[t], SD[t], y[t])
    return {"state": flt.get_state(), "n_obs": int(len(y)), "theta_mean": flt.theta_mean().tolist(),
            "vol_sd": flt.vol_mean(), "ess": flt.ess}


# =============================================================================
# GRADING
# =============================================================================
def total_result(actual_total, line_total):
    if pd.isna(actual_total) or pd.isna(line_total):
        return None
    if actual_total == line_total:
        return "push"
    return "over" if actual_total > line_total else "under"


def spread_result(actual_margin, line_spread_home):
    """line_spread_home is the home team's spread (e.g. -4.5 when home is favored)."""
    if pd.isna(actual_margin) or pd.isna(line_spread_home):
        return None
    net = actual_margin + line_spread_home
    if net == 0:
        return "push"
    return "home_cover" if net > 0 else "away_cover"


def bet_profit(won, price) -> float:
    """Profit in units for a 1-unit stake: won True/False, or None for a push."""
    if won is None or pd.isna(price):
        return 0.0 if won is None else np.nan
    return float(american_to_decimal([price]).iloc[0] - 1.0) if won else -1.0


def clv_points(market: str, side: str, bet_line: float, close_line: float) -> float:
    """Points of closing line value: positive means the line moved toward the bet.
    Totals use the total; spreads use the implied home margin (= -home spread)."""
    if pd.isna(bet_line) or pd.isna(close_line):
        return np.nan
    if market == "total":
        return close_line - bet_line if side == "over" else bet_line - close_line
    return close_line - bet_line if side == "home_cover" else bet_line - close_line


def credible_interval_bounds(conf_level: float) -> tuple[float, float]:
    lower = (100.0 - conf_level * 100.0) / 2.0
    return lower, 100.0 - lower


def side_vs_interval(market_val, lower, upper):
    """HIGH: the line is below the interval (bet over / home); LOW: above (under / away)."""
    if pd.isna(market_val) or pd.isna(lower) or pd.isna(upper):
        return None
    if market_val < lower:
        return "HIGH"
    if market_val > upper:
        return "LOW"
    return "NO_BET"


def quantile_sorted(s: np.ndarray, pct: float) -> float:
    pos = pct / 100.0 * (s.size - 1)
    lo = int(np.floor(pos))
    hi = min(lo + 1, s.size - 1)
    return float(s[lo] + (s[hi] - s[lo]) * (pos - lo))


def prob_above(sorted_draws: np.ndarray | None, line: float) -> float:
    if sorted_draws is None or pd.isna(line):
        return np.nan
    return float(1.0 - np.searchsorted(sorted_draws, line, side="right") / sorted_draws.size)


def prob_below(sorted_draws: np.ndarray | None, line: float) -> float:
    if sorted_draws is None or pd.isna(line):
        return np.nan
    return float(np.searchsorted(sorted_draws, line, side="left") / sorted_draws.size)


KELLY_FRACTION = 0.25   # quarter Kelly
KELLY_CAP = 0.05        # never suggest more than 5% of bankroll on one bet


def ev_choice(sorted_draws: np.ndarray | None, line: float, price_hi, price_lo) -> dict | None:
    """Expected value per 1-unit stake of each side of a line, from the model's draws.

    HIGH = over (totals) / home covers (spreads, line on the implied-home-margin scale);
    LOW = under / away covers. A push returns the stake, so
        EV = P(win) * (decimal - 1) - P(loss).
    Returns the better side with its EV, win probability, the breakeven win rate at that
    price, and a quarter-Kelly stake (fraction of bankroll, capped at KELLY_CAP)."""
    if sorted_draws is None or pd.isna(line):
        return None
    p_hi, p_lo = prob_above(sorted_draws, line), prob_below(sorted_draws, line)
    best = None
    for side, p_win, p_loss, price in (("HIGH", p_hi, p_lo, price_hi), ("LOW", p_lo, p_hi, price_lo)):
        price = price if pd.notna(price) else -110.0
        b = float(american_to_decimal([price]).iloc[0]) - 1.0
        ev = p_win * b - p_loss
        q = p_win / (p_win + p_loss) if p_win + p_loss > 0 else 0.0   # win chance excluding pushes
        kelly = max(0.0, (b * q - (1 - q)) / b) * KELLY_FRACTION if b > 0 else 0.0
        cand = {"side": side, "ev": float(ev), "p_win": float(p_win), "price": float(price),
                "breakeven": float(american_to_prob([price]).iloc[0]), "kelly": float(min(kelly, KELLY_CAP))}
        if best is None or cand["ev"] > best["ev"]:
            best = cand
    return best


def backtest(prep: pd.DataFrame, conf_level: float = 0.55, use_best_price: bool = True,
             rule: str = "interval", min_ev: float = 0.03) -> dict:
    """Money Ball on graded history. Two bet rules:
      interval  bet when the morning line is outside the BPS credible interval (conf_level)
      ev        bet the side whose expected value at the available price is at least min_ev
                (0.03 = 3 cents per unit staked)
    Profit is graded at the price you could have taken (best book by default) and CLV against
    the closing line. prep needs: _total_sorted, _margin_sorted (sorted draws), line_total,
    line_implied_home_margin, actual_total, actual_margin, total_result, spread_result, price
    columns, close_total, close_implied_home_margin."""
    if prep is None or prep.empty:
        return {"empty": True}
    lo_pct, hi_pct = credible_interval_bounds(conf_level)
    out = {"empty": False, "nominal_outside_pct": (1 - conf_level) * 100, "rule": rule}
    rows = []
    markets = [
        ("total", "_total_sorted", "line_total", "actual_total", "total_result", ("over", "under"),
         ("best_over_price", "best_under_price") if use_best_price else ("over_price", "under_price"),
         "close_total"),
        ("spread", "_margin_sorted", "line_implied_home_margin", "actual_margin", "spread_result",
         ("home_cover", "away_cover"),
         ("best_home_price", "best_away_price") if use_best_price else ("home_price", "away_price"),
         "close_implied_home_margin"),
    ]
    for market, draws_col, line_col, actual_col, result_col, (hi_side, lo_side), (hi_px, lo_px), close_col in markets:
        bets, n_usable, outside, n_cov = [], 0, 0, 0
        for _, r in prep.iterrows():
            s = r.get(draws_col)
            if s is None or not isinstance(s, np.ndarray) or s.size == 0:
                continue
            lo, hi = quantile_sorted(s, lo_pct), quantile_sorted(s, hi_pct)
            if pd.notna(r.get(actual_col)):
                n_cov += 1
                outside += int(r[actual_col] < lo or r[actual_col] > hi)
            if rule == "ev":
                choice = ev_choice(s, r.get(line_col), r.get(hi_px), r.get(lo_px))
                call = None if choice is None else (choice["side"] if choice["ev"] >= min_ev else "NO_BET")
                ev = choice["ev"] if choice is not None and call != "NO_BET" else np.nan
            else:
                call, ev = side_vs_interval(r.get(line_col), lo, hi), np.nan
            if call is None:
                continue
            n_usable += 1
            rows.append({"market": market, "game_id": r["game_id"], "call": call, "result": r.get(result_col)})
            if call == "NO_BET":
                continue
            side = hi_side if call == "HIGH" else lo_side
            price = r.get(hi_px if call == "HIGH" else lo_px)
            price = price if pd.notna(price) else -110.0
            result = r.get(result_col)
            won = None if result == "push" else (result == side)
            bets.append({"won": won, "price": price, "profit": bet_profit(won, price), "ev": ev,
                         "clv": clv_points(market, side, r.get(line_col), r.get(close_col)),
                         "breakeven": float(american_to_prob([price]).iloc[0])})
        b = pd.DataFrame(bets)
        graded = b[b["won"].notna()] if len(b) else b
        out[market] = {
            "n_games": n_usable, "n_bets": len(b),
            "selectivity": len(b) / n_usable * 100 if n_usable else 0.0,
            "n_graded": len(graded),
            "win_rate": float(graded["won"].astype(float).mean() * 100) if len(graded) else np.nan,
            "breakeven": float(graded["breakeven"].mean() * 100) if len(graded) else np.nan,
            "units": float(b["profit"].sum()) if len(b) else 0.0,
            "roi": float(b["profit"].sum() / len(b) * 100) if len(b) else np.nan,
            "avg_clv": float(b["clv"].mean()) if len(b) and b["clv"].notna().any() else np.nan,
            "avg_ev": float(b["ev"].mean() * 100) if len(b) and b["ev"].notna().any() else np.nan,
            "outside_pct": outside / n_cov * 100 if n_cov else np.nan,
        }
    out["calls"] = pd.DataFrame(rows, columns=["market", "game_id", "call", "result"])
    return out

## Models A, B and C (each forecasts total and margin)

In [ ]:
"""
The three NBA forecasting models. Each predicts BOTH the game total and the home margin, so
Bayesian Predictive Synthesis can blend all three in each market.

  A  Pace x efficiency   total and margin regressed on the possessions x points-per-possession
                         expectation built from both teams' shrunk pace and ratings, plus rest.
  B  Team strength       every (season, team) gets an offense and defense rating; ratings carry
                         over between seasons with an AR(1) pull toward average. Home and away
                         points share a game-level pace effect (bivariate normal).
  C  Market-anchored     the morning line plus small learned adjustments for how far the pace x
                         efficiency expectation sits from it, and for rest. With no graded lines
                         yet it falls back to its priors: the line itself, with typical NBA error.

Posterior predictive draws are computed in numpy from the saved posterior samples, so the same
code serves any slate of games without rebuilding the PyMC models.
"""


import importlib.util
import logging

import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt

LOG = logging.getLogger("nba_models")

TOTAL_COVS = ["home_b2b", "away_b2b", "home_rest", "away_rest"]
MARGIN_COVS = ["home_b2b", "away_b2b", "rest_diff"]


class Standardizer:
    def __init__(self, cols):
        self.cols, self.mean_, self.std_ = list(cols), None, None

    def fit(self, df):
        self.mean_ = df[self.cols].mean()
        self.std_ = df[self.cols].std().replace(0, 1.0).fillna(1.0)
        return self

    def transform(self, df):
        return ((df[self.cols] - self.mean_) / self.std_).fillna(0.0).to_numpy(float)

    def get_state(self):
        return {"cols": self.cols, "mean_": self.mean_.to_dict(), "std_": self.std_.to_dict()}

    @classmethod
    def from_state(cls, s):
        obj = cls(s["cols"])
        obj.mean_, obj.std_ = pd.Series(s["mean_"]), pd.Series(s["std_"])
        return obj


def _post(idata, name):
    """Posterior samples flattened over chains: shape (draws, ...)."""
    v = idata.posterior[name].values
    return v.reshape((-1,) + v.shape[2:])


def _student_t(rng, nu, size):
    """Standard Student-t draws with per-draw degrees of freedom nu (shape (S,))."""
    z = rng.standard_normal(size)
    g = rng.chisquare(np.broadcast_to(nu[:, None], size))
    return z / np.sqrt(g / nu[:, None])


# =============================================================================
# MODEL A — pace x efficiency
# =============================================================================
def model_a(train: pd.DataFrame, tot_std: Standardizer, mar_std: Standardizer, center: float):
    Zt, Zm = tot_std.transform(train), mar_std.transform(train)
    exp_t = train["exp_total"].to_numpy(float) - center
    exp_m = train["exp_margin"].to_numpy(float)
    with pm.Model() as m:
        a_t = pm.Normal("a_t", 0.0, 5.0)
        b_t = pm.Normal("b_t", 1.0, 0.5)
        g_t = pm.Normal("g_t", 0.0, 2.0, shape=Zt.shape[1])
        a_m = pm.Normal("a_m", 2.0, 2.0)          # home court
        b_m = pm.Normal("b_m", 1.0, 0.5)
        g_m = pm.Normal("g_m", 0.0, 2.0, shape=Zm.shape[1])
        s_t = pm.Gamma("sigma_t", mu=18.0, sigma=5.0)
        s_m = pm.Gamma("sigma_m", mu=12.5, sigma=4.0)
        nu_t = pm.Deterministic("nu_t", 2.0 + pm.Gamma("nu_t_raw", 2.0, 0.1))
        nu_m = pm.Deterministic("nu_m", 2.0 + pm.Gamma("nu_m_raw", 2.0, 0.1))
        pm.StudentT("total", nu=nu_t, mu=center + a_t + b_t * exp_t + pt.dot(Zt, g_t), sigma=s_t,
                    observed=train["total_points"].to_numpy(float))
        pm.StudentT("margin", nu=nu_m, mu=a_m + b_m * exp_m + pt.dot(Zm, g_m), sigma=s_m,
                    observed=train["margin"].to_numpy(float))
    return m


def predict_a(idata, df, tot_std, mar_std, center, rng):
    Zt, Zm = tot_std.transform(df), mar_std.transform(df)
    exp_t = df["exp_total"].to_numpy(float) - center
    exp_m = df["exp_margin"].to_numpy(float)
    mu_t = center + _post(idata, "a_t")[:, None] + _post(idata, "b_t")[:, None] * exp_t + _post(idata, "g_t") @ Zt.T
    mu_m = _post(idata, "a_m")[:, None] + _post(idata, "b_m")[:, None] * exp_m + _post(idata, "g_m") @ Zm.T
    tot = mu_t + _post(idata, "sigma_t")[:, None] * _student_t(rng, _post(idata, "nu_t"), mu_t.shape)
    mar = mu_m + _post(idata, "sigma_m")[:, None] * _student_t(rng, _post(idata, "nu_m"), mu_m.shape)
    return tot, mar


# =============================================================================
# MODEL B — team strength with season-to-season carry-over
# =============================================================================
def team_index(frames: list[pd.DataFrame]) -> dict:
    """(season, team) keys across training and prediction games, ordered by season, plus each
    key's previous-season key (or -1) for the AR(1) carry-over."""
    keys = set()
    for f in frames:
        for side in ("home_team", "away_team"):
            keys.update(zip(f["season"].astype(str), f[side]))
    seasons = sorted({s for s, _ in keys})
    ordered = sorted(keys, key=lambda k: (seasons.index(k[0]), k[1]))
    pos = {k: i for i, k in enumerate(ordered)}
    prev = []
    for s, t in ordered:
        i = seasons.index(s)
        prev.append(pos.get((seasons[i - 1], t), -1) if i > 0 else -1)
    return {"keys": [list(k) for k in ordered], "seasons": seasons, "prev": prev}


def _key_idx(index, df, side):
    pos = {tuple(k): i for i, k in enumerate(index["keys"])}
    return np.array([pos[(str(s), t)] for s, t in zip(df["season"], df[f"{side}_team"])], dtype=int)


def model_b(train: pd.DataFrame, index: dict):
    keys, seasons, prev = index["keys"], index["seasons"], np.array(index["prev"])
    season_of = np.array([seasons.index(k[0]) for k in keys])
    h_idx, a_idx = _key_idx(index, train, "home"), _key_idx(index, train, "away")
    y = np.column_stack([train["home_points"].to_numpy(float), train["away_points"].to_numpy(float)])
    with pm.Model() as m:
        mu = pm.Normal("mu", 115.0, 5.0)
        hca = pm.Normal("hca", 2.0, 1.5)
        rho = pm.Beta("rho", 6.0, 3.0)
        sd_o = pm.HalfNormal("sd_off", 5.0)
        sd_d = pm.HalfNormal("sd_def", 5.0)
        z_o = pm.Normal("z_off", 0.0, 1.0, shape=len(keys))
        z_d = pm.Normal("z_def", 0.0, 1.0, shape=len(keys))
        off, dfn = pt.zeros(len(keys)), pt.zeros(len(keys))
        for s in range(len(seasons)):
            idx = np.where(season_of == s)[0]
            if s == 0:
                off = pt.set_subtensor(off[idx], sd_o * z_o[idx])
                dfn = pt.set_subtensor(dfn[idx], sd_d * z_d[idx])
                continue
            has_prev = prev[idx] >= 0
            p_idx = np.where(has_prev, prev[idx], 0)
            carry_o = pt.switch(has_prev, rho * off[p_idx], 0.0)
            carry_d = pt.switch(has_prev, rho * dfn[p_idx], 0.0)
            scale = pt.sqrt(1.0 - rho ** 2)
            off = pt.set_subtensor(off[idx], carry_o + scale * sd_o * z_o[idx])
            dfn = pt.set_subtensor(dfn[idx], carry_d + scale * sd_d * z_d[idx])
        off = pm.Deterministic("off", off)
        dfn = pm.Deterministic("def", dfn)
        s_e = pm.HalfNormal("sigma_team", 15.0)
        s_g = pm.HalfNormal("sigma_game", 10.0)
        mu_h = mu + hca / 2 + off[h_idx] - dfn[a_idx]
        mu_a = mu - hca / 2 + off[a_idx] - dfn[h_idx]
        var = s_e ** 2 + s_g ** 2
        cov = pt.stack([pt.stack([var, s_g ** 2]), pt.stack([s_g ** 2, var])])
        pm.MvNormal("pts", mu=pt.stack([mu_h, mu_a], axis=1), cov=cov, observed=y)
    return m


def predict_b(idata, df, index, rng):
    h_idx, a_idx = _key_idx(index, df, "home"), _key_idx(index, df, "away")
    off, dfn = _post(idata, "off"), _post(idata, "def")
    mu, hca = _post(idata, "mu")[:, None], _post(idata, "hca")[:, None]
    s_e, s_g = _post(idata, "sigma_team")[:, None], _post(idata, "sigma_game")[:, None]
    mu_h = mu + hca / 2 + off[:, h_idx] - dfn[:, a_idx]
    mu_a = mu - hca / 2 + off[:, a_idx] - dfn[:, h_idx]
    g = s_g * rng.standard_normal(mu_h.shape)
    h = mu_h + g + s_e * rng.standard_normal(mu_h.shape)
    a = mu_a + g + s_e * rng.standard_normal(mu_h.shape)
    return h + a, h - a


# =============================================================================
# MODEL C — market-anchored
# =============================================================================
def model_c(train: pd.DataFrame, tot_std: Standardizer, mar_std: Standardizer):
    """train needs line_total and line_implied_home_margin. Works with zero rows (priors only)."""
    Zt, Zm = tot_std.transform(train), mar_std.transform(train)
    line_t = train["line_total"].to_numpy(float)
    line_m = train["line_implied_home_margin"].to_numpy(float)
    gap_t = train["exp_total"].to_numpy(float) - line_t
    gap_m = train["exp_margin"].to_numpy(float) - line_m
    with pm.Model() as m:
        d0 = pm.Normal("d0", 0.0, 1.5)
        d1 = pm.Normal("d1", 0.0, 0.2)
        g_t = pm.Normal("g_t", 0.0, 0.75, shape=Zt.shape[1])
        e0 = pm.Normal("e0", 0.0, 1.0)
        e1 = pm.Normal("e1", 0.0, 0.2)
        g_m = pm.Normal("g_m", 0.0, 0.75, shape=Zm.shape[1])
        s_t = pm.Gamma("sigma_t", mu=18.0, sigma=3.0)
        s_m = pm.Gamma("sigma_m", mu=12.5, sigma=2.5)
        nu = pm.Deterministic("nu", 2.0 + pm.Gamma("nu_raw", 4.0, 0.2))
        if len(train):
            pm.StudentT("total", nu=nu, mu=line_t + d0 + d1 * gap_t + pt.dot(Zt, g_t), sigma=s_t,
                        observed=train["total_points"].to_numpy(float))
            pm.StudentT("margin", nu=nu, mu=line_m + e0 + e1 * gap_m + pt.dot(Zm, g_m), sigma=s_m,
                        observed=train["margin"].to_numpy(float))
    return m


def predict_c(idata, df, tot_std, mar_std, rng):
    """NaN for games without a line."""
    Zt, Zm = tot_std.transform(df), mar_std.transform(df)
    line_t = df["line_total"].to_numpy(float)
    line_m = df["line_implied_home_margin"].to_numpy(float)
    gap_t = df["exp_total"].to_numpy(float) - line_t
    gap_m = df["exp_margin"].to_numpy(float) - line_m
    mu_t = line_t + _post(idata, "d0")[:, None] + _post(idata, "d1")[:, None] * gap_t + _post(idata, "g_t") @ Zt.T
    mu_m = line_m + _post(idata, "e0")[:, None] + _post(idata, "e1")[:, None] * gap_m + _post(idata, "g_m") @ Zm.T
    nu = _post(idata, "nu")
    tot = mu_t + _post(idata, "sigma_t")[:, None] * _student_t(rng, nu, mu_t.shape)
    mar = mu_m + _post(idata, "sigma_m")[:, None] * _student_t(rng, nu, mu_m.shape)
    return tot, mar


# =============================================================================
# SAMPLING
# =============================================================================
def resolve_sampler(choice: str = "auto") -> str:
    if choice == "auto":
        return "nutpie" if importlib.util.find_spec("nutpie") is not None else "pymc"
    return choice


def sample(model, draws: int, tune: int, chains: int, seed: int, sampler: str = "auto"):
    sampler = resolve_sampler(sampler)
    kw = dict(model=model, draws=draws, tune=tune, chains=chains, progressbar=False, random_seed=seed)
    if sampler != "pymc":
        try:
            return pm.sample(nuts_sampler=sampler, **kw)
        except Exception as e:  # fall back rather than fail a daily run
            LOG.warning("%s sampler failed (%s); falling back to PyMC NUTS.", sampler, e)
    return pm.sample(cores=1, **kw)

## Dashboard: data, pipelines and UI

In [ ]:
"""
NBA Bayesian Betting Dashboard
==============================
The NBA version of the NFL dashboard: three forecasting models, Bayesian Predictive Synthesis
(McAlinn, Aastveit, Nakajima & West 2018) through a Rao-Blackwellized particle filter, Brier
skill against the de-vigged market, confusion matrices, and the Money Ball credible-interval
rule.

Difference from the NFL setup: every model (A pace x efficiency, B team strength, C market-
anchored) forecasts BOTH the total and the home margin, and BPS blends all three in each market.

Data comes from this repo's data/ folder (local clone) or GitHub raw:
  historical_nba_cumulative_stats.csv  pre-game team stats + final scores
  odds_snapshots.csv                   morning and closing lines from every US book

Run in Colab:
  !git clone https://github.com/vladk-III/NBA.git
  %cd NBA
  %run model/nba_dashboard.py
Daily: 1 Train (after the overnight data run) -> 2 Optimize BPS -> 3 Predict.
"""

GITHUB_OWNER = "vladk-III"
GITHUB_REPO = "NBA"
GITHUB_BRANCH = "main"
DATA_SUBDIR = "data"

DRAWS, TUNE, CHAINS = 1000, 1000, 4
NUTS_SAMPLER = "auto"            # "auto" uses nutpie when installed, else PyMC's sampler
TRAIN_SEED = 20261021
BPS_SEED = 20261022
JUPYTER_MODE = "external"
CSV_CACHE_TTL_SEC = 300
MAX_STORED_DRAWS = 1000          # per game per market in predictions_history.csv (file-size guard)
AGENTS = ["A", "B", "C"]
AGENT_NAMES = {"A": "Mod A (Pace×Eff)", "B": "Mod B (Team)", "C": "Mod C (Market)"}

import io
import json
import logging
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo


def _ensure(pkg: str, import_name: str | None = None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)


for _pkg, _imp in [("pymc", "pymc"), ("arviz", "arviz"), ("netCDF4", "netCDF4"), ("requests", "requests"),
                   ("dash", "dash"), ("plotly", "plotly"), ("dash[diskcache]", "diskcache"),
                   ("multiprocess", "multiprocess"), ("nutpie", "nutpie")]:
    try:
        _ensure(_pkg, _imp)
    except subprocess.CalledProcessError:
        pass  # nutpie is optional; the rest will fail loudly on import below

import numpy as np
import pandas as pd
import requests
import arviz as az
import diskcache
import plotly.graph_objects as go
from dash import Dash, html, dcc, Input, Output, State, dash_table, DiskcacheManager

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
LOG = logging.getLogger("nba_dashboard")
EASTERN = ZoneInfo("America/New_York")


def _load_model_modules():
    """Import nba_model_core / nba_models from next to this file, the repo's model/ folder,
    or (when this file was pasted into a notebook) download them from GitHub."""
    here = Path(globals().get("__file__", Path.cwd() / "model" / "x.py")).resolve().parent
    for d in (here, Path.cwd() / "model", Path.cwd()):
        if (d / "nba_model_core.py").exists():
            sys.path.insert(0, str(d))
            break
    else:
        d = Path.cwd() / "_nba_model_src"
        d.mkdir(exist_ok=True)
        for name in ("nba_model_core.py", "nba_models.py"):
            url = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}/model/{name}"
            (d / name).write_text(requests.get(url, timeout=30).text)
        sys.path.insert(0, str(d))
    import nba_model_core as core
    import nba_models as models
    return core, models


if "BPSFilter" in globals() and "model_a" in globals():
    # Single-notebook build (NBA_Dashboard.ipynb): core and model code already live in this namespace.
    core = models = sys.modules[__name__]
else:
    core, models = _load_model_modules()

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_BASE = Path("/content/drive/MyDrive/Stonks") / "Basketball/NBA/2026-27"
except Exception:
    DRIVE_BASE = Path("./NBA_Betting_Engine")
STATE_DIR = DRIVE_BASE / "nba_model_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_HISTORY_PATH = DRIVE_BASE / "predictions_history.csv"
BUNDLE_PATH = STATE_DIR / "model_bundle.pkl"
BPS_BUNDLE_PATH = STATE_DIR / "bps_bundle.pkl"

In [ ]:
# =============================================================================
# THEME & UI STYLES (same autumn palette as the NFL dashboard)
# =============================================================================
theme = {
    "page_bg": "#482E2D", "card_bg": "#FFFDF8", "border": "#E5D9C5", "text_main": "#3F0E00",
    "text_axis": "#8C1C04", "text_muted": "#BF7F04", "grid": "#F4EBE1",
    "colors": ["#8C1C04", "#BF5B05", "#BF7F04", "#92780A", "#3F0E00"],
    "positive": "#92780A", "negative": "#8C1C04",
    "font": "'Segoe UI', 'Trebuchet MS', 'Helvetica Neue', sans-serif",
}
HEATMAP_SCALE = [[0.0, "#FBF1E4"], [0.35, "#EBB98A"], [0.7, "#BF5B05"], [1.0, "#8C1C04"]]
CARD_STYLE = {"backgroundColor": theme["card_bg"], "border": f"2px solid {theme['border']}",
              "borderRadius": "24px", "padding": "24px 28px", "marginBottom": "24px",
              "boxShadow": "0 6px 16px rgba(0, 0, 0, 0.3)"}
BTN_STYLE = {"fontSize": "15px", "padding": "12px 24px", "cursor": "pointer",
             "backgroundColor": theme["colors"][1], "color": "white", "border": "none",
             "borderRadius": "30px", "fontWeight": "700", "marginRight": "14px",
             "boxShadow": "0 2px 8px rgba(191, 91, 5, 0.4)"}
BTN_STYLE_SECONDARY = {**BTN_STYLE, "backgroundColor": "transparent", "color": theme["card_bg"],
                       "border": f"2px solid {theme['card_bg']}", "boxShadow": "none"}
INPUT_STYLE = {"width": "110px", "padding": "8px 12px", "borderRadius": "12px",
               "border": f"2px solid {theme['border']}", "backgroundColor": theme["card_bg"],
               "color": theme["text_main"], "fontWeight": "700", "fontSize": "14px",
               "marginRight": "28px", "fontFamily": theme["font"]}
CONTROL_LABEL_STYLE = {"color": theme["card_bg"], "fontWeight": "700", "marginRight": "10px", "fontSize": "14px"}
TAB_STYLE = {"borderBottom": f"2px solid {theme['colors'][2]}", "padding": "16px 24px", "fontWeight": "700",
             "color": theme["card_bg"], "backgroundColor": "transparent", "borderTop": "none",
             "borderLeft": "none", "borderRight": "none", "fontSize": "16px",
             "borderRadius": "12px 12px 0 0", "opacity": "0.7"}
TAB_SELECTED_STYLE = {**TAB_STYLE, "borderBottom": f"4px solid {theme['colors'][1]}",
                      "color": theme["colors"][1], "opacity": "1.0"}
TABLE_STYLE_CELL = {"fontFamily": theme["font"], "fontSize": "13.5px", "padding": "12px 14px",
                    "backgroundColor": theme["card_bg"], "color": theme["text_main"],
                    "borderBottom": f"1px dashed {theme['grid']}", "textAlign": "left"}
TABLE_STYLE_HEADER = {"fontWeight": "800", "backgroundColor": theme["grid"], "color": theme["colors"][4],
                      "border": "none", "padding": "14px 14px", "textAlign": "left", "fontSize": "12px",
                      "textTransform": "uppercase"}
STATUS_STYLE = {"color": "#E5D9C5", "fontSize": "14px", "fontWeight": "600", "marginBottom": "6px"}


def kpi_card(label, value, sub=None):
    children = [html.Div(label, style={"fontSize": "12.5px", "fontWeight": "700", "color": theme["text_axis"],
                                       "marginBottom": "6px", "textTransform": "uppercase"}),
                html.Div(value, style={"fontSize": "26px", "fontWeight": "800", "color": theme["colors"][0]})]
    if sub:
        children.append(html.Div(sub, style={"fontSize": "12px", "color": theme["text_muted"], "marginTop": "6px"}))
    return html.Div(children, style={**CARD_STYLE, "flex": "1", "minWidth": "190px", "marginBottom": "0px"})


def section_header(title, subtext=None):
    children = [html.H3(title, style={"color": theme["card_bg"], "fontSize": "18px", "fontWeight": "800",
                                      "marginBottom": "2px", "marginTop": "0px"})]
    if subtext:
        children.append(html.P(subtext, style={"fontSize": "13px", "color": theme["border"],
                                               "marginTop": "0px", "marginBottom": "14px"}))
    return html.Div(children)


def empty_state(message, sub=None):
    children = [html.P(message, style={"color": theme["text_main"], "fontSize": "15px", "fontWeight": "700", "margin": "0 0 4px 0"})]
    if sub:
        children.append(html.P(sub, style={"color": theme["text_muted"], "fontSize": "13px", "margin": "0"}))
    return html.Div(children, style={**CARD_STYLE, "textAlign": "center", "padding": "36px 24px"})


def error_state(title, detail):
    return html.Div([html.Div(f"⚠ {title}", style={"color": theme["negative"], "fontWeight": "700", "fontSize": "15px", "marginBottom": "8px"}),
                     html.Pre(detail, style={"fontSize": "12px", "color": theme["text_muted"], "whiteSpace": "pre-wrap", "margin": "0"})],
                    style={**CARD_STYLE, "borderLeft": f"4px solid {theme['negative']}"})


def apply_plotly_theme(fig, title):
    fig.update_layout(title=dict(text=title, font=dict(color=theme["text_main"], family=theme["font"], size=15)),
                      plot_bgcolor=theme["card_bg"], paper_bgcolor=theme["card_bg"],
                      font=dict(color=theme["text_axis"], family=theme["font"], size=12),
                      xaxis=dict(gridcolor=theme["grid"], zerolinecolor=theme["grid"]),
                      yaxis=dict(gridcolor=theme["grid"], zerolinecolor=theme["grid"]),
                      margin=dict(t=50, b=50, l=50, r=24),
                      legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
    return fig


def table(df, page_size=15, cond=None):
    return dash_table.DataTable(data=df.to_dict("records"), columns=[{"name": c, "id": c} for c in df.columns],
                                style_cell=TABLE_STYLE_CELL, style_header=TABLE_STYLE_HEADER, sort_action="native",
                                page_size=page_size, style_data_conditional=cond or [])

In [ ]:
# =============================================================================
# DATA
# =============================================================================
_CSV_CACHE: dict[str, tuple[float, pd.DataFrame]] = {}


def _load_csv(filename: str, max_age: float = CSV_CACHE_TTL_SEC) -> pd.DataFrame:
    """data/<file> from a local clone, else GitHub raw (cached for max_age seconds)."""
    for local in (Path(DATA_SUBDIR) / filename, Path("..") / DATA_SUBDIR / filename):
        if local.exists():
            return pd.read_csv(local, dtype={"game_id": str})
    url = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{DATA_SUBDIR}/{filename}"
    hit = _CSV_CACHE.get(url)
    if hit is not None and max_age > 0 and time.time() - hit[0] < max_age:
        return hit[1].copy()
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            df = pd.read_csv(io.StringIO(r.text), dtype={"game_id": str})
            _CSV_CACHE[url] = (time.time(), df)
            return df.copy()
    except requests.RequestException:
        pass
    return hit[1].copy() if hit is not None else pd.DataFrame()


def load_data(fresh: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    age = 0 if fresh else CSV_CACHE_TTL_SEC
    hist = _load_csv("historical_nba_cumulative_stats.csv", age)
    if hist.empty:
        raise RuntimeError("historical_nba_cumulative_stats.csv is empty or unreachable.")
    return hist, _load_csv("odds_snapshots.csv", age)


def attach_lines(feats: pd.DataFrame, odds: pd.DataFrame) -> pd.DataFrame:
    """Morning consensus lines and prices (closing snapshot as fallback for games with no morning one)."""
    morning = core.market_lines(odds, "morning")
    closing = core.market_lines(odds, "closing")
    lines = pd.concat([morning, closing[~closing["game_id"].isin(morning["game_id"])]], ignore_index=True)
    out = feats.merge(lines, on="game_id", how="left")
    out["line_implied_home_margin"] = -out["line_spread_home"]
    return out


def format_spread_val(val, home, away):
    """Home-margin number -> book notation naming the favorite (e.g. 'BOS -4.5')."""
    if val is None or pd.isna(val):
        return "—"
    v = float(val)
    if abs(v) < 0.05:
        return "EVEN"
    return f"{home} -{abs(v):.1f}" if v > 0 else f"{away} -{abs(v):.1f}"


def fmt_price(p):
    return "—" if p is None or pd.isna(p) else (f"+{int(p)}" if p > 0 else f"{int(p)}")


def _clean(value, default, lo, hi, cast=float):
    try:
        v = cast(round(float(value)) if cast is int else float(value))
    except (TypeError, ValueError):
        return default
    return default if not np.isfinite(v) else max(lo, min(hi, v))

In [ ]:
# =============================================================================
# PIPELINES
# =============================================================================
def run_training_pipeline(set_progress):
    set_progress("Loading historical stats and odds...")
    hist, odds = load_data(fresh=True)
    final = hist["home_points"].notna()
    prior = core.league_prior(hist[final])
    feats = attach_lines(core.build_features(hist, prior), odds)
    train = feats[feats["total_points"].notna()].reset_index(drop=True)
    upcoming = feats[feats["total_points"].isna()]
    if train.empty:
        raise RuntimeError("No completed games to train on.")
    train_c = train.dropna(subset=["line_total", "line_implied_home_margin"]).reset_index(drop=True)

    tot_std = models.Standardizer(models.TOTAL_COVS).fit(train)
    mar_std = models.Standardizer(models.MARGIN_COVS).fit(train)
    center = float(train["exp_total"].mean())
    # Every team of the latest season gets a rating slot, so tomorrow's games can be predicted
    # without retraining even when a team has not appeared in an upcoming row yet.
    latest = sorted(feats["season"].unique())[-1]
    teams = sorted(set(feats["home_team"]) | set(feats["away_team"]))
    all_latest = pd.DataFrame({"season": latest, "home_team": teams, "away_team": teams})
    index = models.team_index([train, upcoming, all_latest])
    sampler = models.resolve_sampler(NUTS_SAMPLER)

    set_progress(f"Training Model A (pace × efficiency) on {len(train)} games with {sampler} NUTS...")
    idata_a = models.sample(models.model_a(train, tot_std, mar_std, center), DRAWS, TUNE, CHAINS, TRAIN_SEED, NUTS_SAMPLER)
    idata_a.to_netcdf(str(STATE_DIR / "model_a.nc"))
    set_progress(f"Training Model B (team strength, {len(index['keys'])} team-seasons)...")
    idata_b = models.sample(models.model_b(train, index), DRAWS, TUNE, CHAINS, TRAIN_SEED + 1, NUTS_SAMPLER)
    idata_b.to_netcdf(str(STATE_DIR / "model_b.nc"))
    set_progress(f"Training Model C (market-anchored) on {len(train_c)} games with lines"
                 + (" — priors only until graded lines exist" if train_c.empty else "") + "...")
    idata_c = models.sample(models.model_c(train_c, tot_std, mar_std), DRAWS, TUNE, CHAINS, TRAIN_SEED + 2, NUTS_SAMPLER)
    idata_c.to_netcdf(str(STATE_DIR / "model_c.nc"))

    with open(BUNDLE_PATH, "wb") as f:
        pickle.dump({"prior": prior, "tot_std": tot_std.get_state(), "mar_std": mar_std.get_state(),
                     "center": center, "index": index, "n_train": len(train), "n_train_c": len(train_c),
                     "trained_at": pd.Timestamp.now(tz="UTC").isoformat()}, f)
    set_progress(f"Training complete: {len(train)} games (A, B), {len(train_c)} with lines (C).")
    return {"ok": True}


def load_bps(set_progress, n_particles, delta, beta):
    if BPS_BUNDLE_PATH.exists():
        try:
            with open(BPS_BUNDLE_PATH, "rb") as f:
                b = pickle.load(f)
            ft = core.BPSFilter.from_state(b["total"]["state"], seed=BPS_SEED + 1)
            fm = core.BPSFilter.from_state(b["margin"]["state"], seed=BPS_SEED + 2)
            tt, tm = b["total"]["theta_mean"], b["margin"]["theta_mean"]
            set_progress(f"Loaded BPS (p={ft.n}). Totals θ=[{tt[0]:.2f}, A={tt[1]:.2f}, B={tt[2]:.2f}, C={tt[3]:.2f}] "
                         f"Margins θ=[{tm[0]:.2f}, A={tm[1]:.2f}, B={tm[2]:.2f}, C={tm[3]:.2f}]")
            return ft, fm
        except Exception as e:
            LOG.warning("Could not load BPS bundle (%s); using prior filters.", e)
    set_progress(f"No fitted BPS yet — equal-weight (⅓ each) synthesis with p={n_particles}. "
                 "Run step 2 once predicted games are graded.")
    return (core.BPSFilter(3, n_particles, delta, beta, core.BPS_FALLBACK_OBS_VAR["total"], BPS_SEED + 1),
            core.BPSFilter(3, n_particles, delta, beta, core.BPS_FALLBACK_OBS_VAR["margin"], BPS_SEED + 2))


def run_bps_pipeline(set_progress, n_particles, delta, beta):
    n_particles = _clean(n_particles, core.BPS_DEFAULTS["particles"], 50, 20000, int)
    delta = _clean(delta, core.BPS_DEFAULTS["delta"], 0.5, 1.0)
    beta = _clean(beta, core.BPS_DEFAULTS["beta"], 0.5, 1.0)
    if not PREDICTIONS_HISTORY_PATH.exists():
        raise RuntimeError("No predictions_history.csv yet. Run predictions for a few days and let the games finish.")
    hist, odds = load_data(fresh=True)
    graded = prepare_graded_history(pd.read_csv(PREDICTIONS_HISTORY_PATH, dtype={"game_id": str}), hist, odds)
    if graded is None or graded.empty:
        raise RuntimeError("None of the logged predictions have final scores yet.")
    set_progress(f"{len(graded)} graded predictions. Filtering totals (p={n_particles}, δ={delta:g}, β={beta:g})...")
    tot = core.fit_bps(graded, [f"{a}_total_mean" for a in AGENTS], [f"{a}_total_std" for a in AGENTS],
                       "actual_total", n_particles, delta, beta, BPS_SEED + 1)
    set_progress("Filtering margins...")
    mar = core.fit_bps(graded, [f"{a}_margin_mean" for a in AGENTS], [f"{a}_margin_std" for a in AGENTS],
                       "actual_margin", n_particles, delta, beta, BPS_SEED + 2)
    with open(BPS_BUNDLE_PATH, "wb") as f:
        pickle.dump({"total": tot, "margin": mar, "n_particles": n_particles, "delta": delta, "beta": beta}, f)
    tt, tm = tot["theta_mean"], mar["theta_mean"]
    set_progress(f"BPS complete. Totals θ: int={tt[0]:.2f}, A={tt[1]:.2f}, B={tt[2]:.2f}, C={tt[3]:.2f}, "
                 f"σ={tot['vol_sd']:.1f} (n={tot['n_obs']}, ESS={tot['ess']:.0f}) | Margins θ: int={tm[0]:.2f}, "
                 f"A={tm[1]:.2f}, B={tm[2]:.2f}, C={tm[3]:.2f}, σ={mar['vol_sd']:.1f} (n={mar['n_obs']})")
    return {"ok": True}


def edge_tier(z):
    if pd.isna(z):
        return "No Line"
    a = abs(z)
    return "★★★ Strong" if a >= 1.5 else "★★☆ Moderate" if a >= 0.75 else "★☆☆ Slight"


HISTORY_COLS = (["game_id", "game_date", "season", "home_team", "away_team"]
                + [f"{a}_{m}_{s}" for a in AGENTS for m in ("total", "margin") for s in ("mean", "std")]
                + ["combined_total_mean", "combined_total_median", "combined_total_std",
                   "combined_margin_mean", "combined_margin_median", "combined_margin_std",
                   "line_total", "line_spread_home", "line_implied_home_margin",
                   "over_price", "under_price", "best_over_price", "best_under_price",
                   "home_price", "away_price", "best_home_price", "best_away_price",
                   "edge_total", "edge_margin", "edge_total_z", "edge_margin_z", "total_tier", "spread_tier",
                   "model_prob_over", "model_prob_home_cover", "total_draws_json", "margin_draws_json", "predicted_at"])


def run_predictions_pipeline(set_progress, game_date, n_samples, n_particles, delta, beta):
    n_samples = _clean(n_samples, core.BPS_DEFAULTS["samples"], 200, 100000, int)
    n_particles = _clean(n_particles, core.BPS_DEFAULTS["particles"], 50, 20000, int)
    delta = _clean(delta, core.BPS_DEFAULTS["delta"], 0.5, 1.0)
    beta = _clean(beta, core.BPS_DEFAULTS["beta"], 0.5, 1.0)
    if not BUNDLE_PATH.exists():
        raise RuntimeError("No trained models yet. Run step 1 first.")
    with open(BUNDLE_PATH, "rb") as f:
        bundle = pickle.load(f)
    tot_std = models.Standardizer.from_state(bundle["tot_std"])
    mar_std = models.Standardizer.from_state(bundle["mar_std"])

    set_progress("Loading today's matchups and lines...")
    hist, odds = load_data(fresh=True)
    feats = attach_lines(core.build_features(hist, bundle["prior"]), odds)
    open_games = feats[feats["total_points"].isna()]
    today = datetime.now(EASTERN).strftime("%Y-%m-%d")
    target = str(game_date)[:10] if game_date else today
    slate = open_games[open_games["game_date"] == target]
    if slate.empty:
        later = sorted(d for d in open_games["game_date"].unique() if d >= today)
        if not later:
            raise RuntimeError(f"No unplayed games on {target}. The data pipeline adds each day's games overnight.")
        target = later[0]
        slate = open_games[open_games["game_date"] == target]
    slate = slate.reset_index(drop=True)
    known = {tuple(k) for k in bundle["index"]["keys"]}
    unknown = [f"{a}@{h}" for s, h, a in zip(slate["season"], slate["home_team"], slate["away_team"])
               if (s, h) not in known or (s, a) not in known]
    if unknown:
        raise RuntimeError(f"Teams not in the trained model ({', '.join(unknown)}). Re-run step 1.")

    rng = np.random.default_rng(BPS_SEED + int(pd.Timestamp(target).strftime("%Y%m%d")))
    set_progress(f"Forecasting {len(slate)} games on {target} with Models A, B and C...")
    draws = {
        "A": models.predict_a(az.from_netcdf(str(STATE_DIR / "model_a.nc")), slate, tot_std, mar_std, bundle["center"], rng),
        "B": models.predict_b(az.from_netcdf(str(STATE_DIR / "model_b.nc")), slate, bundle["index"], rng),
        "C": models.predict_c(az.from_netcdf(str(STATE_DIR / "model_c.nc")), slate, tot_std, mar_std, rng),
    }
    out = slate[["game_id", "game_date", "season", "home_team", "away_team", "line_total", "line_spread_home",
                 "line_implied_home_margin", "over_price", "under_price", "best_over_price", "best_under_price",
                 "home_price", "away_price", "best_home_price", "best_away_price"]].copy()
    for a, (tot, mar) in draws.items():
        out[f"{a}_total_mean"], out[f"{a}_total_std"] = tot.mean(0), tot.std(0)
        out[f"{a}_margin_mean"], out[f"{a}_margin_std"] = mar.mean(0), mar.std(0)

    flt_t, flt_m = load_bps(set_progress, n_particles, delta, beta)
    set_progress(f"Synthesizing BPS draws ({n_samples:,} per game per market)...")
    recs = []
    for i, r in out.iterrows():
        mu_t = [r[f"{a}_total_mean"] for a in AGENTS]
        mu_m = [r[f"{a}_margin_mean"] for a in AGENTS]
        rec = {}
        if np.all(np.isfinite(mu_t)) and np.all(np.isfinite(mu_m)):
            pool_t = flt_t.predict(mu_t, [r[f"{a}_total_std"] for a in AGENTS], n_samples)["draws"]
            pool_m = flt_m.predict(mu_m, [r[f"{a}_margin_std"] for a in AGENTS], n_samples)["draws"]
            for name, pool, line in (("total", pool_t, r["line_total"]), ("margin", pool_m, r["line_implied_home_margin"])):
                rec[f"combined_{name}_mean"] = float(pool.mean())
                rec[f"combined_{name}_median"] = float(np.median(pool))
                rec[f"combined_{name}_std"] = float(pool.std())
                rec[f"edge_{name}"] = float(pool.mean() - line) if pd.notna(line) else np.nan
                rec[f"edge_{name}_z"] = rec[f"edge_{name}"] / float(pool.std()) if pd.notna(line) else np.nan
                rec[f"{name}_draws_json"] = json.dumps(np.round(pool[:MAX_STORED_DRAWS], 1).tolist())
            rec["model_prob_over"] = float((pool_t > r["line_total"]).mean()) if pd.notna(r["line_total"]) else np.nan
            rec["model_prob_home_cover"] = (float((pool_m > r["line_implied_home_margin"]).mean())
                                            if pd.notna(r["line_implied_home_margin"]) else np.nan)
        recs.append(rec)
    out = pd.concat([out, pd.DataFrame(recs, index=out.index)], axis=1)
    for c in HISTORY_COLS:
        if c not in out.columns:
            out[c] = np.nan
    out["total_tier"] = out["edge_total_z"].apply(edge_tier)
    out["spread_tier"] = out["edge_margin_z"].apply(edge_tier)
    out["predicted_at"] = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ")

    set_progress("Saving prediction history...")
    new = out[HISTORY_COLS]
    if PREDICTIONS_HISTORY_PATH.exists():
        old = pd.read_csv(PREDICTIONS_HISTORY_PATH, dtype={"game_id": str})
        new = pd.concat([old, new], ignore_index=True).sort_values("predicted_at").drop_duplicates("game_id", keep="last")
    new.to_csv(PREDICTIONS_HISTORY_PATH, index=False)
    n_blend = int(out["combined_total_mean"].notna().sum())
    set_progress(f"Predictions complete for {target}: {len(out)} games, {n_blend} with a line (BPS blend needs Model C, which needs the line).")
    return {"ok": True, "date": target, "data": out.to_json(orient="records")}

In [ ]:
# =============================================================================
# GRADING
# =============================================================================
_GRADED_CACHE: dict = {}


def _sorted_draws(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)) or raw == "":
        return None
    try:
        a = np.asarray(json.loads(raw), dtype=float)
    except (ValueError, TypeError):
        return None
    a = a[np.isfinite(a)]
    return np.sort(a) if a.size else None


def prepare_graded_history(history: pd.DataFrame, hist: pd.DataFrame, odds: pd.DataFrame) -> pd.DataFrame | None:
    """Prediction log + final scores + closing lines + results, with draws parsed once."""
    if history is None or history.empty:
        return None
    key = (int(pd.util.hash_pandas_object(history, index=False).sum()), len(hist), len(odds))
    if key in _GRADED_CACHE:
        return _GRADED_CACHE[key]
    finals = hist.loc[hist["home_points"].notna(), ["game_id", "home_points", "away_points"]].copy()
    finals["game_id"] = finals["game_id"].astype(str)
    close = core.market_lines(odds, "closing")[["game_id", "line_total", "line_spread_home"]]
    close = close.rename(columns={"line_total": "close_total", "line_spread_home": "close_spread_home"})
    h = history.copy()
    h["game_id"] = h["game_id"].astype(str)
    g = h.merge(finals, on="game_id", how="inner").merge(close, on="game_id", how="left")
    if g.empty:
        _GRADED_CACHE[key] = None
        return None
    g["close_implied_home_margin"] = -g["close_spread_home"]
    g["actual_total"] = g["home_points"] + g["away_points"]
    g["actual_margin"] = g["home_points"] - g["away_points"]
    g["total_result"] = [core.total_result(a, l) for a, l in zip(g["actual_total"], g["line_total"])]
    g["spread_result"] = [core.spread_result(a, l) for a, l in zip(g["actual_margin"], g["line_spread_home"])]
    g["_total_sorted"] = g["total_draws_json"].map(_sorted_draws)
    g["_margin_sorted"] = g["margin_draws_json"].map(_sorted_draws)
    g = g.reset_index(drop=True)
    if len(_GRADED_CACHE) > 8:
        _GRADED_CACHE.clear()
    _GRADED_CACHE[key] = g
    return g


def directional_confusion(df, pred_col, actual_col, line_col):
    """2x2 of predicted side vs actual side of the book line; pushes and on-the-line excluded."""
    if any(c not in df.columns for c in (pred_col, actual_col, line_col)):
        return {"cm": np.zeros((2, 2), dtype=int), "n": 0, "pushes": 0, "no_call": 0}
    v = df.dropna(subset=[pred_col, actual_col, line_col])
    pushes = int((v[actual_col] == v[line_col]).sum())
    v = v[v[actual_col] != v[line_col]]
    no_call = int((v[pred_col] == v[line_col]).sum())
    v = v[v[pred_col] != v[line_col]]
    p_hi, a_hi = (v[pred_col] > v[line_col]).to_numpy(), (v[actual_col] > v[line_col]).to_numpy()
    cm = np.array([[np.sum(a_hi & p_hi), np.sum(a_hi & ~p_hi)], [np.sum(~a_hi & p_hi), np.sum(~a_hi & ~p_hi)]], dtype=int)
    return {"cm": cm, "n": int(cm.sum()), "pushes": pushes, "no_call": no_call}


def confusion_heatmap(cm, x_labels, y_labels, title, x_title="Model call", y_title="Actual result"):
    cm = np.asarray(cm, dtype=float)
    col_tot = cm.sum(axis=0, keepdims=True)
    col_pct = np.divide(cm, col_tot, out=np.full_like(cm, np.nan), where=col_tot > 0) * 100
    zmax = max(cm.max(), 1.0)
    fig = go.Figure(go.Heatmap(z=cm, x=x_labels, y=y_labels, zmin=0, zmax=zmax, colorscale=HEATMAP_SCALE,
                               showscale=False, xgap=2, ygap=2, customdata=col_pct,
                               hovertemplate="Actual: %{y}<br>Call: %{x}<br><b>%{z:.0f} games</b><br>%{customdata:.1f}% of this call<extra></extra>"))
    for i, y in enumerate(y_labels):
        for j, x in enumerate(x_labels):
            dark = cm[i, j] / zmax > 0.55
            pct = f"{col_pct[i, j]:.0f}% of {x} calls" if np.isfinite(col_pct[i, j]) else "—"
            fig.add_annotation(x=x, y=y, showarrow=False, text=f"<b>{int(cm[i, j])}</b><br><span style='font-size:11px'>{pct}</span>",
                               font=dict(size=18, color=theme["card_bg"] if dark else theme["text_main"], family=theme["font"]))
    apply_plotly_theme(fig, title)
    fig.update_layout(height=340, margin=dict(t=56, b=56, l=110, r=24))
    fig.update_xaxes(title_text=x_title, showgrid=False, type="category")
    fig.update_yaxes(title_text=y_title, autorange="reversed", showgrid=False, type="category")
    return fig


CM_MODEL_OPTIONS = {
    "bps_median": ("BPS Blend (Median)", "combined_total_median", "combined_margin_median"),
    "bps_mean": ("BPS Blend (Mean)", "combined_total_mean", "combined_margin_mean"),
    **{f"agent_{a}": (AGENT_NAMES[a], f"{a}_total_mean", f"{a}_margin_mean") for a in AGENTS},
}


def confusion_section(graded, model_key):
    label, tot_col, mar_col = CM_MODEL_OPTIONS.get(model_key, CM_MODEL_OPTIONS["bps_median"])
    if graded is None or graded.empty:
        return empty_state("No graded games yet.", "Confusion matrices appear once predicted games have final scores.")
    panels = []
    for res, title, labels in [
        (directional_confusion(graded, tot_col, "actual_total", "line_total"), "Totals — Over / Under", ["Over", "Under"]),
        (directional_confusion(graded, mar_col, "actual_margin", "line_implied_home_margin"), "Spreads — Against the Spread", ["Home covers", "Away covers"]),
    ]:
        cm, n = res["cm"], res["n"]
        hits = cm[0, 0] + cm[1, 1]
        summary = f"Hit rate {hits / n * 100:.1f}% of {n}" if n else "No graded games with a line."
        if res["pushes"] or res["no_call"]:
            summary += f" · excluded: {res['pushes']} pushes, {res['no_call']} on the line"
        panels.append(html.Div([dcc.Graph(figure=confusion_heatmap(cm, labels, labels, title), config={"displayModeBar": False}),
                                html.P(summary, style={"fontSize": "12.5px", "color": theme["text_main"], "margin": "4px 0 0 0"})],
                               style={"flex": 1, "minWidth": "340px", "padding": "16px", "backgroundColor": theme["card_bg"], "borderRadius": "12px"}))
    return html.Div([html.H4(f"Calls vs. Book — {label}", style={"color": theme["colors"][4], "fontSize": "15px", "margin": "0 0 12px 0"}),
                     html.Div(panels, style={"display": "flex", "gap": "20px", "flexWrap": "wrap"})],
                    style={"padding": "20px", "backgroundColor": theme["grid"], "borderRadius": "12px", "marginBottom": "32px"})


def brier_skill(graded, p_col, result_col, yes, no, price_yes, price_no):
    """Brier skill of the model's probability vs the de-vigged market from that game's own prices."""
    if graded is None or graded.empty or p_col not in graded.columns:
        return np.nan, 0
    g = graded[graded[result_col].isin([yes, no])].copy()
    g["p_mkt"] = core.devig_pair(g[price_yes].fillna(-110), g[price_no].fillna(-110)).values
    g = g.dropna(subset=[p_col, "p_mkt"])
    if g.empty:
        return np.nan, 0
    y = (g[result_col] == yes).astype(float)
    base = float(((g["p_mkt"] - y) ** 2).mean())
    return (1 - float(((g[p_col] - y) ** 2).mean()) / base if base > 0 else np.nan), len(g)


def moneyball_sweep(prep, rule="interval"):
    """Backtest across interval levels (interval rule) or EV thresholds (EV rule)."""
    rows = []
    levels = np.arange(0.50, 0.8001, 0.025) if rule == "interval" else np.arange(0.0, 0.1001, 0.01)
    for lvl in levels:
        res = (core.backtest(prep, float(lvl)) if rule == "interval"
               else core.backtest(prep, rule="ev", min_ev=float(lvl)))
        if res.get("empty"):
            continue
        rows.append({"Level": lvl * 100,
                     **{f"{m.title()} {k}": res[m][key] for m in ("total", "spread")
                        for k, key in (("Selectivity", "selectivity"), ("Win Rate", "win_rate"),
                                       ("ROI", "roi"), ("Breakeven", "breakeven"))}})
    return pd.DataFrame(rows)


def sweep_figure(sweep, market, level, rule="interval"):
    m = market.title()
    fig = go.Figure()
    for col, name, color in ((f"{m} Selectivity", "Selectivity (% of games bet)", theme["colors"][1]),
                             (f"{m} Win Rate", "Win rate (%)", theme["colors"][4]),
                             (f"{m} ROI", "ROI (%)", theme["colors"][3])):
        fig.add_trace(go.Scatter(x=sweep["Level"], y=sweep[col], name=name, mode="lines+markers",
                                 line=dict(color=color, width=2), marker=dict(size=7)))
    be = sweep[f"{m} Breakeven"].dropna()
    fig.add_hline(y=float(be.mean()) if len(be) else 52.38, line_dash="dot", line_color=theme["negative"],
                  annotation_text="Breakeven win rate at the prices taken")
    fig.add_hline(y=0, line_color=theme["grid"])
    fig.add_vline(x=level * 100, line_dash="dash", line_color=theme["colors"][4], annotation_text="Current")
    x_title = "Credible interval level (%)" if rule == "interval" else "Minimum EV per unit staked (%)"
    fig.update_layout(xaxis_title=x_title, yaxis=dict(title="%"), hovermode="x unified")
    what = "width" if rule == "interval" else "EV threshold"
    return apply_plotly_theme(fig, f"{'Totals' if market == 'total' else 'Spreads'} — results by {what}").update_layout(height=360)

In [ ]:
# =============================================================================
# DASH APP
# =============================================================================
app = Dash(__name__, background_callback_manager=DiskcacheManager(diskcache.Cache(str(Path("/tmp/nba_dash_cache")))),
           suppress_callback_exceptions=True, title="NBA Projections")
app.index_string = f"""<!DOCTYPE html><html><head>{{%metas%}}<title>{{%title%}}</title>{{%favicon%}}{{%css%}}
<style>html, body {{ background-color: {theme['page_bg']}; margin: 0 !important; padding: 0 !important; }}
* {{ box-sizing: border-box; }}</style></head><body>{{%app_entry%}}<footer>{{%config%}}{{%scripts%}}{{%renderer%}}</footer></body></html>"""


def _num_input(id_, value, lo, hi, step):
    return dcc.Input(id=id_, type="number", min=lo, max=hi, step=step, value=value, debounce=True, style=INPUT_STYLE)


tab1_layout = html.Div([
    html.Div([html.Button("1 · Train Models", id="train-btn", n_clicks=0, style=BTN_STYLE),
              html.Button("2 · Optimize BPS Mixture", id="bps-btn", n_clicks=0, style=BTN_STYLE),
              html.Button("3 · Predict Slate", id="predict-btn", n_clicks=0, style=BTN_STYLE_SECONDARY)],
             style={"marginBottom": "16px"}),
    html.Div([html.Span("Game date: ", style=CONTROL_LABEL_STYLE),
              dcc.Input(id="pred-date", type="text", placeholder="today (YYYY-MM-DD)", debounce=True,
                        style={**INPUT_STYLE, "width": "170px"}),
              html.Span("BPS particles (p): ", style=CONTROL_LABEL_STYLE),
              _num_input("bps-particles", core.BPS_DEFAULTS["particles"], 50, 20000, 50),
              html.Span("State discount (δ): ", style=CONTROL_LABEL_STYLE),
              _num_input("bps-delta", core.BPS_DEFAULTS["delta"], 0.5, 1.0, 0.005),
              html.Span("Volatility discount (β): ", style=CONTROL_LABEL_STYLE),
              _num_input("bps-beta", core.BPS_DEFAULTS["beta"], 0.5, 1.0, 0.005),
              html.Span("Draws per game: ", style=CONTROL_LABEL_STYLE),
              _num_input("pred-samples", core.BPS_DEFAULTS["samples"], 200, 100000, 100)],
             style={"marginBottom": "16px", "display": "flex", "flexWrap": "wrap", "alignItems": "center", "rowGap": "10px"}),
    html.Div([html.Div(id=i, style=STATUS_STYLE) for i in ("train-status", "bps-status", "predict-status")]),
    html.Div([html.Span("Projection basis: ", style=CONTROL_LABEL_STYLE),
              dcc.RadioItems(id="proj-basis-toggle", value="median", inline=True,
                             options=[{"label": " Median", "value": "median"}, {"label": " Mean", "value": "mean"}],
                             labelStyle={"color": theme["card_bg"], "marginRight": "20px", "fontWeight": "600"})],
             style={"margin": "12px 0 20px 0"}),
    dcc.Store(id="train-store", storage_type="session"),
    dcc.Store(id="bps-store", storage_type="session"),
    dcc.Store(id="predictions-store", storage_type="session"),
    html.Div(id="dash-results"),
])

tab2_layout = html.Div([
    html.Div([html.Button("Refresh Ledger", id="refresh-history-btn", n_clicks=0, style=BTN_STYLE),
              html.Span("Confusion matrix model: ", style={**CONTROL_LABEL_STYLE, "marginLeft": "12px"}),
              dcc.Dropdown(id="cm-model-dropdown", value="bps_median", clearable=False, searchable=False,
                           options=[{"label": v[0], "value": k} for k, v in CM_MODEL_OPTIONS.items()],
                           style={"width": "300px", "color": theme["text_main"]})],
             style={"display": "flex", "alignItems": "center", "flexWrap": "wrap", "gap": "8px"}),
    dcc.Loading(html.Div(id="performance-results", style={"marginTop": "24px"})),
])

tab3_layout = html.Div([
    html.Div([
        html.Div([html.Label("Bet rule", style={"fontWeight": "700", "marginRight": "16px", "color": theme["text_main"]}),
                  dcc.RadioItems(id="moneyball-rule", value="ev", inline=True,
                                 options=[{"label": " Expected value (price-aware)", "value": "ev"},
                                          {"label": " Credible interval (original)", "value": "interval"}],
                                 labelStyle={"marginRight": "24px", "fontWeight": "600", "color": theme["text_main"]})],
                 style={"marginBottom": "18px"}),
        html.Div([html.Label("Minimum EV per unit staked", style={"fontWeight": "700", "color": theme["text_main"]}),
                  dcc.Slider(id="moneyball-ev-slider", min=0, max=10, step=0.5, value=3,
                             marks={v: f"{v}%" for v in range(0, 11, 2)}, tooltip={"placement": "bottom", "always_visible": True})],
                 id="ev-slider-box", style={"marginBottom": "12px"}),
        html.Div([html.Label("Credible interval width", style={"fontWeight": "700", "color": theme["text_main"]}),
                  dcc.Slider(id="moneyball-ci-slider", min=50, max=80, step=2.5, value=55,
                             marks={v: f"{v}%" for v in range(50, 81, 5)}, tooltip={"placement": "bottom", "always_visible": True})],
                 id="ci-slider-box", style={"display": "none"}),
    ], style={**CARD_STYLE, "padding": "18px 28px 30px 28px"}),
    dcc.Loading(html.Div(id="tab3-results")),
])

PAGE = {"padding": "36px 40px"}
app.layout = html.Div([
    html.Div([html.H1("NBA Projections & Insight Ledger", style={"margin": "0", "color": "#FFFDF8", "fontSize": "28px", "fontWeight": "800"}),
              html.P("Three models, Bayesian Predictive Synthesis, and skill tracking against the market.",
                     style={"margin": "6px 0 0 0", "color": theme["border"], "fontSize": "15px", "fontWeight": "600"})],
             style={"padding": "32px 40px", "marginBottom": "12px"}),
    dcc.Tabs(id="tabs", value="tab1", children=[
        dcc.Tab(label="Forecast & Models", value="tab1", style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE),
        dcc.Tab(label="Skill & Evaluation", value="tab2", style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE),
        dcc.Tab(label="💰 Money Ball", value="tab3", style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE)]),
    html.Div(tab1_layout, id="tab1-page", style={**PAGE, "display": "block"}),
    html.Div(tab2_layout, id="tab2-page", style={**PAGE, "display": "none"}),
    html.Div(tab3_layout, id="tab3-page", style={**PAGE, "display": "none"}),
], style={"fontFamily": theme["font"], "backgroundColor": theme["page_bg"], "minHeight": "100vh"})


@app.callback(Output("tab1-page", "style"), Output("tab2-page", "style"), Output("tab3-page", "style"), Input("tabs", "value"))
def toggle_tabs(tab):
    return tuple({**PAGE, "display": "block" if tab == t else "none"} for t in ("tab1", "tab2", "tab3"))


def _background(fn, label):
    def run(set_progress, *args):
        try:
            return fn(lambda msg: set_progress([msg]), *args)
        except Exception as e:
            LOG.exception("%s failed", label)
            set_progress([f"⚠ {label} failed: {e}"])
            return {"ok": False, "error": str(e)}
    return run


@app.callback(Output("train-store", "data"), Input("train-btn", "n_clicks"), background=True,
              running=[(Output("train-btn", "disabled"), True, False)], progress=[Output("train-status", "children")],
              prevent_initial_call=True)
def train_callback(set_progress, _):
    return _background(run_training_pipeline, "Training")(set_progress)


@app.callback(Output("bps-store", "data"), Input("bps-btn", "n_clicks"),
              State("bps-particles", "value"), State("bps-delta", "value"), State("bps-beta", "value"),
              background=True, running=[(Output("bps-btn", "disabled"), True, False)],
              progress=[Output("bps-status", "children")], prevent_initial_call=True)
def bps_callback(set_progress, _, p, d, b):
    return _background(run_bps_pipeline, "BPS")(set_progress, p, d, b)


@app.callback(Output("predictions-store", "data"), Input("predict-btn", "n_clicks"),
              State("pred-date", "value"), State("pred-samples", "value"), State("bps-particles", "value"),
              State("bps-delta", "value"), State("bps-beta", "value"),
              background=True, running=[(Output("predict-btn", "disabled"), True, False)],
              progress=[Output("predict-status", "children")], prevent_initial_call=True)
def predict_callback(set_progress, _, gdate, n, p, d, b):
    return _background(run_predictions_pipeline, "Prediction")(set_progress, gdate, n, p, d, b)


@app.callback(Output("dash-results", "children"), Input("predictions-store", "data"), Input("proj-basis-toggle", "value"))
def render_results(store, basis):
    if not store:
        return ""
    if not store.get("ok"):
        return error_state("Prediction pipeline failed", store.get("error", "Unknown error"))
    df = pd.read_json(io.StringIO(store["data"]), orient="records", dtype={"game_id": str})
    if df.empty:
        return empty_state("No games to show.")
    df["Game"] = df["away_team"] + " @ " + df["home_team"]
    tot_col, mar_col = f"combined_total_{basis}", f"combined_margin_{basis}"
    tot = pd.DataFrame({"Game": df["Game"],
                        **{AGENT_NAMES[a]: df[f"{a}_total_mean"].round(1) for a in AGENTS},
                        f"BPS ({basis.title()})": df[tot_col].round(1),
                        "Line": df["line_total"],
                        "P(Over)": (df["model_prob_over"] * 100).round(1),
                        "Best Over / Under": [f"{fmt_price(o)} / {fmt_price(u)}" for o, u in zip(df["best_over_price"], df["best_under_price"])],
                        "Edge": df["total_tier"]})
    spr = pd.DataFrame({"Game": df["Game"],
                        **{AGENT_NAMES[a]: [format_spread_val(v, h, w) for v, h, w in zip(df[f"{a}_margin_mean"], df["home_team"], df["away_team"])] for a in AGENTS},
                        f"BPS ({basis.title()})": [format_spread_val(v, h, w) for v, h, w in zip(df[mar_col], df["home_team"], df["away_team"])],
                        "Line": [format_spread_val(v, h, w) for v, h, w in zip(df["line_implied_home_margin"], df["home_team"], df["away_team"])],
                        "P(Home Covers)": (df["model_prob_home_cover"] * 100).round(1),
                        "Best Home / Away": [f"{fmt_price(o)} / {fmt_price(u)}" for o, u in zip(df["best_home_price"], df["best_away_price"])],
                        "Edge": df["spread_tier"]})
    bold = [{"if": {"column_id": f"BPS ({basis.title()})"}, "fontWeight": "bold", "backgroundColor": "#f9f3ea"}]
    return html.Div([
        html.H3(f"Totals — {store.get('date', '')}", style={"color": theme["card_bg"], "marginBottom": "8px"}),
        html.Div(table(tot, cond=bold), style={"marginBottom": "32px", "overflowX": "auto"}),
        html.H3("Spreads", style={"color": theme["card_bg"], "marginBottom": "8px"}),
        html.P("Each model forecasts both markets. Spreads name the favorite (BOS -4.5 = Boston by 4.5).",
               style={"color": theme["border"], "fontSize": "13px", "marginTop": "0"}),
        html.Div(table(spr, cond=bold), style={"overflowX": "auto"}),
    ])


def _accuracy_rows(graded, market):
    actual, line = ("actual_total", "line_total") if market == "total" else ("actual_margin", "line_implied_home_margin")
    rows = []
    for name, col in [(AGENT_NAMES[a], f"{a}_{market}_mean") for a in AGENTS] + \
                     [("BPS Blend (Mean)", f"combined_{market}_mean"), ("BPS Blend (Median)", f"combined_{market}_median")]:
        v = graded.dropna(subset=[col, actual]) if col in graded.columns else graded.iloc[0:0]
        res = directional_confusion(graded, col, actual, line)
        win = (res["cm"][0, 0] + res["cm"][1, 1]) / res["n"] * 100 if res["n"] else np.nan
        rows.append({"Model": name,
                     "MAE": round(float((v[col] - v[actual]).abs().mean()), 2) if len(v) else "—",
                     "Bias": round(float((v[col] - v[actual]).mean()), 2) if len(v) else "—",
                     "Win % vs line": f"{win:.1f}%" if pd.notna(win) else "—", "Games": len(v)})
    return pd.DataFrame(rows)


@app.callback(Output("performance-results", "children"), Input("refresh-history-btn", "n_clicks"), Input("cm-model-dropdown", "value"))
def render_performance(n_clicks, cm_model):
    if not PREDICTIONS_HISTORY_PATH.exists():
        return empty_state("No prediction history yet.", "It builds up each time you run predictions.")
    history = pd.read_csv(PREDICTIONS_HISTORY_PATH, dtype={"game_id": str})
    hist, odds = load_data(fresh=(n_clicks or 0) > 0)
    graded = prepare_graded_history(history, hist, odds)
    if graded is None or graded.empty:
        return empty_state("No graded games yet.", f"{len(history)} predictions logged; they're graded once final scores land.")
    bss_t, n_t = brier_skill(graded, "model_prob_over", "total_result", "over", "under", "over_price", "under_price")
    bss_s, n_s = brier_skill(graded, "model_prob_home_cover", "spread_result", "home_cover", "away_cover", "home_price", "away_price")
    bold = [{"if": {"filter_query": '{Model} contains "BPS"'}, "fontWeight": "bold", "backgroundColor": "#f9f3ea"}]
    arch = graded.sort_values("game_date", ascending=False)
    archive = pd.DataFrame({
        "Date": arch["game_date"], "Game": arch["away_team"] + " @ " + arch["home_team"],
        "Book Total": arch["line_total"], "Model Total": arch["combined_total_median"].round(1),
        "Actual Total": arch["actual_total"], "Close Total": arch["close_total"],
        "Book Line": [format_spread_val(v, h, a) for v, h, a in zip(arch["line_implied_home_margin"], arch["home_team"], arch["away_team"])],
        "Model Line": [format_spread_val(v, h, a) for v, h, a in zip(arch["combined_margin_median"], arch["home_team"], arch["away_team"])],
        "Actual": [format_spread_val(v, h, a) for v, h, a in zip(arch["actual_margin"], arch["home_team"], arch["away_team"])]})
    return html.Div([
        html.H3("Performance Ledger", style={"color": theme["colors"][0], "marginTop": 0, "fontWeight": "800"}),
        html.Div([kpi_card("Totals Brier Skill", f"{bss_t:.4f}" if pd.notna(bss_t) else "—",
                           f"P(over) vs de-vigged market at each game's prices · {n_t} games"),
                  kpi_card("Spreads Brier Skill", f"{bss_s:.4f}" if pd.notna(bss_s) else "—",
                           f"P(home covers) vs de-vigged market · {n_s} games")],
                 style={"display": "flex", "gap": "24px", "marginBottom": "24px", "flexWrap": "wrap"}),
        html.P("Above 0 means the model's probabilities beat the market's own (vig removed using both sides' prices).",
               style={"fontSize": "12.5px", "color": theme["text_muted"], "marginTop": "-12px"}),
        confusion_section(graded, cm_model),
        html.Div([html.Div([html.H4("Totals accuracy by model", style={"color": theme["colors"][4], "marginTop": 0}),
                            table(_accuracy_rows(graded, "total"), cond=bold)], style={"flex": 1, "minWidth": "320px"}),
                  html.Div([html.H4("Spreads accuracy by model", style={"color": theme["colors"][4], "marginTop": 0}),
                            table(_accuracy_rows(graded, "margin"), cond=bold)], style={"flex": 1, "minWidth": "320px"})],
                 style={"display": "flex", "gap": "20px", "marginBottom": "32px", "flexWrap": "wrap"}),
        html.H4("Game-by-game archive", style={"color": theme["colors"][4]}),
        table(archive),
    ], style=CARD_STYLE)


def moneyball_live(out, market, rule, conf, min_ev):
    """Today's calls under the selected rule. HIGH = over / home covers; LOW = under / away."""
    lo_pct, hi_pct = core.credible_interval_bounds(conf)
    total = market == "total"
    rows = []
    for _, r in out.iterrows():
        draws = _sorted_draws(r.get(f"{market}_draws_json"))
        line = r["line_total"] if total else r["line_implied_home_margin"]
        px_hi, px_lo = (r["best_over_price"], r["best_under_price"]) if total else (r["best_home_price"], r["best_away_price"])
        fmt = (lambda v: round(v, 1) if pd.notna(v) else None) if total else \
              (lambda v, h=r["home_team"], a=r["away_team"]: format_spread_val(v, h, a))
        row = {"Game": f"{r['away_team']} @ {r['home_team']}", "Book Line": fmt(line)}
        if rule == "ev":
            c = core.ev_choice(draws, line, px_hi, px_lo)
            side = None if c is None else (c["side"] if c["ev"] >= min_ev else "NO_BET")
            lean = None if c is None else c["side"]
            row.update({"Lean": "—" if lean is None else (("OVER" if lean == "HIGH" else "UNDER") if total
                                                           else (r["home_team"] if lean == "HIGH" else r["away_team"])),
                        "P(Win)": f"{c['p_win'] * 100:.1f}%" if c else "—",
                        "Breakeven": f"{c['breakeven'] * 100:.1f}%" if c else "—",
                        "EV / unit": f"{c['ev'] * 100:+.1f}%" if c else "—",
                        "Best Price": fmt_price(c["price"]) if c else "—",
                        "¼ Kelly stake": f"{c['kelly'] * 100:.1f}% of bankroll" if c and side not in (None, "NO_BET") else "—"})
        else:
            lo = core.quantile_sorted(draws, lo_pct) if draws is not None else np.nan
            hi = core.quantile_sorted(draws, hi_pct) if draws is not None else np.nan
            side = core.side_vs_interval(line, lo, hi)
            row.update({f"{conf * 100:.0f}% CrI Lower": fmt(lo), f"{conf * 100:.0f}% CrI Upper": fmt(hi)})
        if side is None:
            call, price = "No Line", np.nan
        elif side == "NO_BET":
            call, price = "NO BET", np.nan
        else:
            call = ("OVER" if side == "HIGH" else "UNDER") if total else (r["home_team"] if side == "HIGH" else r["away_team"])
            price = px_hi if side == "HIGH" else px_lo
        row["Call"] = call
        if rule != "ev":
            row["Best Price"] = fmt_price(price)
        rows.append(row)
    return pd.DataFrame(rows)


@app.callback(Output("ev-slider-box", "style"), Output("ci-slider-box", "style"), Input("moneyball-rule", "value"))
def toggle_rule_controls(rule):
    show, hide = {"marginBottom": "12px"}, {"display": "none"}
    return (show, hide) if rule == "ev" else (hide, show)


@app.callback(Output("tab3-results", "children"), Input("moneyball-rule", "value"), Input("moneyball-ci-slider", "value"),
              Input("moneyball-ev-slider", "value"), Input("predictions-store", "data"))
def render_moneyball(rule, ci_pct, ev_pct, store):
    conf, min_ev = ci_pct / 100.0, ev_pct / 100.0
    lo_pct, hi_pct = core.credible_interval_bounds(conf)
    if rule == "ev":
        intro = section_header("💰 Money Ball — Expected value", f"Bet a side only when its expected profit is at least {ev_pct:g}% of the stake.")
        explain = (f"For each side, the BPS draws give the chance of winning, losing and pushing at the morning line. "
                   f"EV per unit = P(win) × payout − P(loss), at the best price across books. The rule bets the better side "
                   f"when its EV is at least {ev_pct:g}%. A side needs a win chance above its breakeven (52.4% at −110) just to "
                   "reach 0%. The quarter-Kelly stake is a sizing guide, capped at 5% of bankroll. EV is only as good as the "
                   "model's probabilities, so check the Brier skill on the Skill tab before trusting it.")
        rule_label, current = f"EV ≥ {ev_pct:g}%", min_ev
    else:
        intro = section_header("💰 Money Ball — Credible interval", "Bet only when the line falls OUTSIDE the BPS credible interval.")
        explain = (f"The {ci_pct:.0f}% interval spans the {lo_pct:.2f}th to {hi_pct:.2f}th percentile of the BPS draws. "
                   "Line below it: OVER / home side; above it: UNDER / away side; inside: NO BET. NBA game-to-game noise "
                   "makes this interval wide, so this rule bets rarely.")
        rule_label, current = f"{ci_pct:.0f}% interval", conf
    sections = [intro, html.P(explain, style={"fontSize": "13px", "color": theme["border"], "marginBottom": "24px", "maxWidth": "900px"}),
                section_header("Today's calls", f"{rule_label} vs. the morning line")]
    call_style = [{"if": {"filter_query": '{Call} = "NO BET"'}, "color": theme["text_muted"]},
                  {"if": {"filter_query": '{Call} != "NO BET" && {Call} != "No Line"'}, "fontWeight": "bold", "color": theme["positive"]}]
    if not store or not store.get("ok"):
        sections.append(empty_state("No live predictions loaded.", "Run step 3 on the Forecast & Models tab."))
    else:
        out = pd.read_json(io.StringIO(store["data"]), orient="records", dtype={"game_id": str})
        for market, label in (("total", "Totals"), ("margin", "Spreads")):
            sections.append(html.Div(label, style={"fontWeight": "700", "marginBottom": "8px", "color": theme["card_bg"]}))
            sections.append(html.Div(table(moneyball_live(out, market, rule, conf, min_ev), 10, call_style), style={"marginBottom": "20px"}))

    sections.append(section_header("Historical backtest", f"The {rule_label} rule on graded history, flat 1-unit bets"))
    if not PREDICTIONS_HISTORY_PATH.exists():
        sections.append(empty_state("No prediction history yet."))
        return html.Div(sections)
    hist, odds = load_data(fresh=False)
    prep = prepare_graded_history(pd.read_csv(PREDICTIONS_HISTORY_PATH, dtype={"game_id": str}), hist, odds)
    bt = core.backtest(prep, conf, rule=rule, min_ev=min_ev)
    if bt.get("empty"):
        sections.append(empty_state("No graded games yet.", "Results appear once predicted games are final."))
        return html.Div(sections)
    cards = []
    for m, label in (("total", "Totals"), ("spread", "Spreads")):
        r = bt[m]
        ev_note = f" · avg expected {r['avg_ev']:+.1f}%" if rule == "ev" and pd.notna(r["avg_ev"]) else ""
        cards += [kpi_card(f"{label} — Bets", f"{r['n_bets']} / {r['n_games']}", f"Selectivity {r['selectivity']:.1f}%"),
                  kpi_card(f"{label} — Win Rate", f"{r['win_rate']:.1f}%" if pd.notna(r["win_rate"]) else "—",
                           f"breakeven {r['breakeven']:.1f}% at prices taken" if pd.notna(r["breakeven"]) else ""),
                  kpi_card(f"{label} — Units", f"{r['units']:+.2f}", (f"ROI {r['roi']:+.1f}%" if pd.notna(r["roi"]) else "") + ev_note),
                  kpi_card(f"{label} — CLV", f"{r['avg_clv']:+.2f} pts" if pd.notna(r["avg_clv"]) else "—",
                           f"interval misses {r['outside_pct']:.0f}% vs {bt['nominal_outside_pct']:.0f}% nominal"
                           if rule == "interval" and pd.notna(r["outside_pct"]) else "")]
    sections.append(html.Div(cards, style={"display": "flex", "gap": "20px", "flexWrap": "wrap", "marginBottom": "20px"}))
    sections.append(html.P("CLV = points the line moved toward your bet between the morning and closing snapshots. "
                           "Consistently positive CLV is the earliest sign of a real edge, well before win rate settles. "
                           + ("With the EV rule, compare realized ROI to the average expected EV: if ROI keeps landing well below it, "
                              "the model's probabilities are overconfident." if rule == "ev" else ""),
                           style={"fontSize": "12.5px", "color": theme["border"], "fontStyle": "italic", "maxWidth": "860px"}))
    calls = bt["calls"]
    figs = []
    for m, (hi, lo), (c_hi, c_lo), ylab in (("total", ("over", "under"), ("OVER", "UNDER"), ["Over", "Under"]),
                                             ("spread", ("home_cover", "away_cover"), ("Bet home", "Bet away"), ["Home covered", "Away covered"])):
        d = calls[(calls["market"] == m) & calls["result"].isin([hi, lo])]
        cm = np.array([[np.sum((d["call"] == s) & (d["result"] == res)) for s in ("HIGH", "NO_BET", "LOW")] for res in (hi, lo)])
        figs.append(confusion_heatmap(cm, [c_hi, "NO BET", c_lo], ylab, f"{'Totals' if m == 'total' else 'Spreads'} — {rule_label} vs. result", x_title="Money Ball call"))
    sections.append(html.Div([html.Div(dcc.Graph(figure=f, config={"displayModeBar": False}), style={"flex": 1, "minWidth": "380px", **CARD_STYLE}) for f in figs],
                             style={"display": "flex", "gap": "20px", "flexWrap": "wrap"}))
    sweep = moneyball_sweep(prep, rule)
    if not sweep.empty:
        sections.append(section_header("Finding the right setting",
                                       "Selectivity, win rate and ROI across " + ("EV thresholds" if rule == "ev" else "interval levels")))
        sections.append(html.Div([html.Div(dcc.Graph(figure=sweep_figure(sweep, m, current, rule)), style={"flex": 1, "minWidth": "380px", **CARD_STYLE})
                                  for m in ("total", "spread")], style={"display": "flex", "gap": "20px", "flexWrap": "wrap"}))
    return html.Div(sections)


if __name__ == "__main__":
    app.run(jupyter_mode=JUPYTER_MODE, debug=False) if "ipykernel" in sys.modules else app.run(debug=False, port=int(os.environ.get("PORT", 8050)))